# Dummy tests

In [23]:
import sys
import os.path
sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))))))

from pylondrina.issues.core import IssueSpec, emit_issue

In [63]:
DUMMY_ISSUES = {
  # 1) message fijo + details_keys + defaults
  "TST.FIXED.DEFAULTS": IssueSpec(
      code="TST.FIXED.DEFAULTS",
      level="error",
      message_template="Mensaje fijo.",
      details_keys=("schema_version", "fields_size", "note"),
      defaults={"note": "default_note"},
  ),

  # 2) template con placeholders + !r
  "TST.TEMPLATE.REPR": IssueSpec(
      code="TST.TEMPLATE.REPR",
      level="warning",
      message_template="Campo {field!r} rule {rule!r}.",
      details_keys=("field", "rule"),
  ),

  # 3) build_details callable
  "TST.DETAILS.BUILDER": IssueSpec(
      code="TST.DETAILS.BUILDER",
      level="info",
      message_template="x={x}",
      build_details=lambda ctx: {"double": ctx["x"] * 2},
  ),

  # 4) sin details (details_keys vacío)
  "TST.NO.DETAILS": IssueSpec(
      code="TST.NO.DETAILS",
      level="warning",
      message_template="Sin details.",
  ),
  # 5) Para testear empty details (details_keys con keys pero no se pasan en emit_issue)
  "TST.EMPTY.DETAILS": IssueSpec(
    code="TST.EMPTY.DETAILS",
    level="warning",
    message_template="Empty details test.",
    details_keys=("x",),  # pero no pasaremos x
  ),
  "TST.USES.FIELD": IssueSpec(
    code="TST.USES.FIELD",
    level="info",
    message_template="Field={field}, Source={source_field}, Rows={row_count}",
  ),

}

## Test A: Code desconocido

Qué prueba: que no se cuelen typos.

Esperado: ValueError

In [32]:
issues = []

try:
    emit_issue(issues, DUMMY_ISSUES, "NO.EXISTE_ESTE_CODIGO")
except ValueError as error:
    print(error)

print("\nElementos en issues: ", len(issues))

Unknown issue code: NO.EXISTE_ESTE_CODIGO

Elementos en issues:  0


## Test B: Template simple sin placeholders

Qué prueba: message fijo y details por keys.

In [30]:
issues = []
emit_issue(issues, DUMMY_ISSUES, "TST.FIXED.DEFAULTS", schema_version="1.1", fields_size=0)
print(len(issues))
print(issues[0],"\n")

assert issues[-1].message == "Mensaje fijo."
assert issues[-1].details == {'schema_version': '1.1', 'fields_size': 0, 'note': 'default_note'}

1
Issue(level='error', code='TST.FIXED.DEFAULTS', message='Mensaje fijo.', field=None, source_field=None, row_count=None, details={'schema_version': '1.1', 'fields_size': 0, 'note': 'default_note'}) 



## Test C: Template con placeholders + !r

Qué prueba: formateo correcto tipo "El campo {field!r}...".

In [35]:
issues = []
emit_issue(issues, DUMMY_ISSUES, "TST.TEMPLATE.REPR", field="mode", rule="pattern")

assert issues[-1].message == "Campo 'mode' rule 'pattern'."
assert issues[-1].details == {'field': 'mode', 'rule': 'pattern'}

## Test D: Defaults en IssueSpec

Qué prueba: defaults se mezclan con ctx.

In [ ]:
issues = []

emit_issue(issues, DUMMY_ISSUES, "TST.FIXED.DEFAULTS", )
assert issues[-1].details["note"] == "default_note"

# Pueba override
emit_issue(issues, DUMMY_ISSUES, "TST.FIXED.DEFAULTS", note="otro")
assert issues[-1].details["note"] == "otro"

## Test E: message_override debe ganar al template

In [ ]:
issues = []

emit_issue(issues, DUMMY_ISSUES, "TST.FIXED.DEFAULTS")
assert issues[-1].message == "Mensaje fijo."

# Override del mensaje
emit_issue(issues, DUMMY_ISSUES, "TST.FIXED.DEFAULTS", message_override="MSG")
assert issues[-1].message == "MSG"

## Test F: details explícito debe ganar sobre details_keys/build_details

In [44]:
issues = []

emit_issue(issues, DUMMY_ISSUES, "TST.TEMPLATE.REPR", field="mode", rule="pattern")
assert issues[-1].details == {'field': 'mode', 'rule': 'pattern'}

emit_issue(issues, DUMMY_ISSUES, "TST.TEMPLATE.REPR",field="mode", rule="pattern", details={"a":1})
assert issues[-1].details == {"a":1}

## Test G: build_details (callable) se usa cuando existe

In [51]:
issues = []
emit_issue(issues, DUMMY_ISSUES, "TST.DETAILS.BUILDER", x = 3)

assert issues[-1].message == "x=3"
assert issues[-1].details == {"double":6}

## Test H: details vacíos → None

Objetivo: si details_keys no encuentra ninguna key presente en ctx, details debe ser None (no {}).

In [62]:
issues = []
issue = emit_issue(issues, DUMMY_ISSUES, "TST.EMPTY.DETAILS")

assert issue is issues[-1]

## Test I: field / source_field / row_count se propagan

Verifica que emit_issue copia correctamente esos tres atributos al Issue final. Esto importa porque:
- son parte del “contrato” del Issue (no solo details).
- muchas veces la lógica (resúmenes, filtros por campo, conteos) dependerá de ellos.

In [64]:
issues = []
issue = emit_issue(
    issues,
    DUMMY_ISSUES,
    "TST.NO.DETAILS",
    field="mode",
    source_field="modo",
    row_count=7,
)

assert issue.field == "mode"
assert issue.source_field == "modo"
assert issue.row_count == 7

Para además verificar “y también van en ctx” (si wl template los usa), se usa otro spec dummy:

In [65]:
issues = []
issue = emit_issue(
    issues, DUMMY_ISSUES, "TST.USES.FIELD",
    field="mode", source_field="modo", row_count=7
)
assert issue.message == "Field=mode, Source=modo, Rows=7"

## Test J: Append y orden

Objetivo: cada emit_issue agrega al final y retorna el mismo objeto que quedó al final.

In [66]:
issues = []
issue1 = emit_issue(issues, DUMMY_ISSUES, "TST.NO.DETAILS")
assert len(issues) == 1
assert issue1 is issues[-1]

issue2 = emit_issue(issues, DUMMY_ISSUES, "TST.TEMPLATE.REPR", field="mode", rule="pattern")
assert len(issues) == 2
assert issue2 is issues[-1]
assert issues[0].code == "TST.NO.DETAILS"
assert issues[1].code == "TST.TEMPLATE.REPR"